In [1]:
import datetime
import calendar

# Obtener la fecha actual
hoy = datetime.date.today()

# Calcular el primer día del mes actual y restar un día para obtener el mes anterior
primer_dia_mes_actual = hoy.replace(day=1)
ultimo_dia_mes_anterior = primer_dia_mes_actual - datetime.timedelta(days=1)

# Extraer mes y año del mes anterior
mes_anterior = ultimo_dia_mes_anterior.month
anio_anterior = ultimo_dia_mes_anterior.year

# Asignar variables
MES_SQL_SP_LIQ_ACTUAL = f"'{anio_anterior}{mes_anterior:02d}'"
PERIODO_ALMACEN = f"{anio_anterior}{mes_anterior:02d}"
PERIODO_NORMAL_float = f"{anio_anterior}{mes_anterior:02d}"

# Imprimir resultados
print("SP =", MES_SQL_SP_LIQ_ACTUAL)
print("PERIODO_FINANCIERO =", PERIODO_ALMACEN)
print("PERIODO_NORMAL_float =", PERIODO_NORMAL_float)

SP = '202610'
PERIODO_FINANCIERO = 202610
PERIODO_NORMAL_float = 202610


In [2]:
#Importacion de librerias
import pandas as pd
import numpy as np
from pandas import json_normalize
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
from unidecode import unidecode
from googleapiclient.discovery import build
from google.oauth2 import service_account

In [3]:
# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)

Conexion a la base de datos realizada


In [4]:
#CONSULTA PARA CREAR LA TABLA TEMPORAL EN SQL 
with engine.connect() as connection:
    try:

        sql_query = text(f'''
         IF object_id('CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_CONTEO_VENTAS') IS NOT NULL
            DROP TABLE CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_CONTEO_VENTAS


        Delete from Comisiones_Contact.FIJA_LIQ_ACTUAL_DESGLOS where Periodo_Pago = {MES_SQL_SP_LIQ_ACTUAL}
        Delete from Comisiones_Contact.PREGR_APROB_CONTAC_MOD2 where Periodo_Pago_1 = {MES_SQL_SP_LIQ_ACTUAL}
        Delete from Comisiones_Contact.POSG_APROB_CONTAC_MOD2 where Periodo_Pago_1 = {MES_SQL_SP_LIQ_ACTUAL}
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito


In [5]:
#ACA CONSULTO SOLO MIS ETIQUETAS OK PARA EXCLUSIVAMENTE SACAR LA TABLA DE CONTEOS
#-------------------------------------------------------------------------------
host2_8 = '172.16.1.33'
port2_8 = '1433'
database2_8= 'CUN_REPOSITORIO'

try:
    url2_8 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_8, port=port2_8, db=database2_8)
    engine2_8= create_engine(url2_8) 
    print("Conexion a la base de datos realizada")
    Etiquetas_mes_acutual = pd.read_sql(f'''
select * from COMISIONES_CONTACT.ETIQUETAS where Periodo_Pago ={MES_SQL_SP_LIQ_ACTUAL} and EstadoFinal = 'OK'
                            ''', engine2_8)
    Etiquetas_mes_acutual2 = Etiquetas_mes_acutual.copy()
        
except Exception as e:
    print("Error:", e)
finally:
    if 'engine2_8' in locals():
        engine2_8.dispose()
        print("Conexión cerrada")
Etiquetas_mes_acutual['Llave_etiq_per']=Etiquetas_mes_acutual['Doc_Estudiante'].astype(str)+Etiquetas_mes_acutual['PERIODO'].astype(str)
Etiquetas_mes_acutual.to_excel('etiqutas_iniciales_orig.xlsx')
Etiquetas_mes_acutual

Conexion a la base de datos realizada
Conexión cerrada


,Doc_Estudiante,PERIODO,Documento_del_Asesor,Fecha_de_pago_Real,Propietario_del_registro,Programa_vendido,Fecha_de_pago,Correo_Asesor,Periodo_Pago,EST_MATRICULADO,...,TIPO,ORDEN,VAL_LIQUIDADO,NUEVO,ESTADO_PAGO,Documento_Estudiante_zoho,Motivo,Propietario_de_Fase_Name,EstadoFinal,Llave_etiq_per
0,1001046458,26ES6,1000519887,28/10/2026,Contact Danna Valentina Molano Suarez,202610_ESPECIALIZACIÓN EN INTELIGENCIA ARTIFIC...,28/10/2026,DANNA_MOLANO@CUN.EDU.CO,202610.0,1-Activo,...,Postgrado,3966408.0,6598013.0,NUEVO,PAGO,1001046458,ok,None,OK,100104645826ES6
1,1002153533,2027A,1010170664,09/10/2026,Contact Juan Sebastian Mancera Cordero,202610_ADMINISTRACIÓN DE LA SEGURIDAD Y SALUD ...,09/10/2026,JUAN_MANCERAC@CUN.EDU.CO,202610.0,1-Activo,...,Pregrado,3952713.0,2776000.0,NUEVO,PAGO,1002153533,ok,None,OK,10021535332027A
2,1002163454,26V06,1022441171,20/10/2026,Contact Jonathan Daniel Gonzalez Barragan,202610_INGENIERIA DE SISTEMAS,20/10/2026,JONATHAN_GONZALEZBA@CUN.EDU.CO,202610.0,1-Activo,...,Pregrado,3964395.0,3854800.0,NUEVO,PAGO,1002163454,ok,None,OK,100216345426V06
3,1001050522,2027A,1003188298,06/10/2026,Contact Lenis Cecilia Carvajal Cifuentes,202610_DIRECCION Y PRODUCCION DE MEDIOS AUDIOV...,06/10/2026,LENIS_CARVAJAL@CUN.EDU.CO,202610.0,1-Activo,...,Pregrado,3946805.0,3484500.0,NUEVO,PAGO,1001050522,ok,None,OK,10010505222027A
4,1000226882,26V06,1018460372,24/10/2026,Contact Jose Leonardo Galeano Osuna,202610_INGENIERIA INDUSTRIAL,24/10/2026,JOSE_GALEANO@CUN.EDU.CO,202610.0,1-Activo,...,Pregrado,3968226.0,3854800.0,NUEVO,PAGO,1000226882,ok,None,OK,100022688226V06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3421,96601727,26ES6,1053765963,29/10/2026,Contact Santiago Gomez Lopez,202610_ESPECIALIZACIÓN EN ALTA GERENCIA,29/10/2026,SANTIAGO_GOMEZL@CUN.EDU.CO,202610.0,1-Activo,...,Postgrado,3972309.0,6598000.0,NUEVO,PAGO,96601727,ok,None,OK,9660172726ES6
3422,976287,26V06,1000004014,05/10/2026,Contact Alisson Daniza Ramirez Galan,202610_MARKETING DIGITAL,05/10/2026,ALISSON_RAMIREZG@CUN.EDU.CO,202610.0,1-Activo,...,Pregrado,3946795.0,4037200.0,NUEVO,PAGO,976287,ok,None,OK,97628726V06
3423,98400070,26V06,1022441171,09/10/2026,Contact Jonathan Daniel Gonzalez Barragan,202610_MARKETING DIGITAL,09/10/2026,JONATHAN_GONZALEZBA@CUN.EDU.CO,202610.0,1-Activo,...,Pregrado,3957282.0,4037200.0,NUEVO,PAGO,98400070,ok,None,OK,9840007026V06
3424,98468588,26V06,1023371151,07/10/2026,Contact Johanna Valentina Torres Arias,202610_ADMINISTRACIÓN DE EMPRESAS,07/10/2026,JOHANNA_TORRESA@CUN.EDU.CO,202610.0,1-Activo,...,Pregrado,3916435.0,3672400.0,NUEVO,PAGO,98468588,ok,None,OK,9846858826V06


In [6]:
#EN ESTOS PASOS TOMO LAS CEDULAS UNICAS DE ESTUDIANTE PARA FILTRARLAS EN LOS RECIBOS DE CAJA - ME TOCO DIVIDIRLAS
#SI EN TOTAL SON 2500 CEDULA,  YA PROBE Y SI EN cedulas_1 hubiesen 500 cedulas si funciona
#------------------------------------------------------------------------------------------
import numpy as np
Cedulas_filtro = Etiquetas_mes_acutual['Doc_Estudiante'].unique()
num_partes = 10
size = len(Cedulas_filtro) // num_partes
residual = len(Cedulas_filtro) % num_partes
cedulas_partes = []
start = 0
for i in range(num_partes):
    end = start + size + (1 if i < residual else 0)
    cedulas_partes.append(Cedulas_filtro[start:end])
    start = end
cedulas_1, cedulas_2, cedulas_3, cedulas_4, cedulas_5, cedulas_5_1, cedulas_7, cedulas_8, cedulas_9, cedulas_10 = cedulas_partes
#----------------------------------------------------------------------------------
Cedulas_filtro1 = cedulas_1
cedulas_consulta1 = ", ".join([f"''{cedula1}''" for cedula1 in Cedulas_filtro1])
Cedulas_filtro2 = cedulas_2
cedulas_consulta2 = ", ".join([f"''{cedula2}''" for cedula2 in Cedulas_filtro2])
Cedulas_filtro3 = cedulas_3
cedulas_consulta3= ", ".join([f"''{cedula3}''" for cedula3 in Cedulas_filtro3])
Cedulas_filtro4 = cedulas_4
cedulas_consulta4= ", ".join([f"''{cedula4}''" for cedula4 in Cedulas_filtro4])
Cedulas_filtro5 = cedulas_5
cedulas_consulta5= ", ".join([f"''{cedula5}''" for cedula5 in Cedulas_filtro5])
Cedulas_filtro5_1 = cedulas_5_1
cedulas_consulta5_1= ", ".join([f"''{cedula5_1}''" for cedula5_1 in Cedulas_filtro5_1])
Cedulas_filtro7 = cedulas_7
cedulas_consulta7= ", ".join([f"''{cedula7}''" for cedula7 in Cedulas_filtro7])
Cedulas_filtro8 = cedulas_8
cedulas_consulta8= ", ".join([f"''{cedula8}''" for cedula8 in Cedulas_filtro8])
Cedulas_filtro9 = cedulas_9
cedulas_consulta9= ", ".join([f"''{cedula9}''" for cedula9 in Cedulas_filtro9])
Cedulas_filtro10 = cedulas_10
cedulas_consulta10= ", ".join([f"''{cedula10}''" for cedula10 in Cedulas_filtro10])
#------------------------------------------------------------------------------------
cedulas_dict = {1: cedulas_consulta1,2: cedulas_consulta2,3: cedulas_consulta3,4: cedulas_consulta4,5: cedulas_consulta5,6: cedulas_consulta5_1,7: cedulas_consulta7,8: cedulas_consulta8,9: cedulas_consulta9,10: cedulas_consulta10}



In [7]:
#ACA FILTRO LAS CEDULAS DE ESTUDIANTE EN LOS RECIBOS DE CAJA (TOCO CON UN FOR) YA ESTA FUNCIONAL
#-------------------------------------------------------------------------------
import time
Recibos_Caja = pd.DataFrame()
for i in range(1, 11):
    cedulas_consulta = cedulas_dict[i]
    host2_9 = '172.16.1.33'
    port2_9 = '1433'
    database2_9= 'CUN_REPOSITORIO'
    try:
        url2_9 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_9, port=port2_9, db=database2_9)
        engine2_9= create_engine(url2_9) 
        print("Conexion a la base de datos realizada")
        df = pd.read_sql(f'''
            SELECT * FROM OPENQUERY([172.16.1.175],
            'SELECT * FROM iceberg.recibo_caja WHERE CLIENTE IN ({cedulas_consulta}) AND ESTADO = ''V'';')
                                ''', engine2_9)
        Recibos_Caja = pd.concat([Recibos_Caja, df], ignore_index=True)
    except Exception as e:
        print("Error:", e)
    finally:
        if 'engine2_9' in locals():
            engine2_9.dispose()
            print("Conexión cerrada")
    time.sleep(2)
Recibos_Caja

Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada
Conexion a la base de datos realizada
Conexión cerrada


,ORGANIZACION,NUMERO,CAJA,SECUENCIA_PERSONA,FECHA,DESCRIPCION,VALOR,ESTADO,CLIENTE,FECHA_PAGO,PERIODO,CONSECUTIVO
0,1,561299175.0,201,592282.0,2026-11-05 00:00:00.0000000,Recibo Automatico por Pago en BOTON DE PAGOS d...,161000.0,V,1.000120e+09,2026-11-05 00:00:00.0000000,26V06,2873443.0
1,1,382597337.0,103,592282.0,2026-10-27 00:00:00.0000000,Cruce AUTOMATICO DE SALDOS -VIRTUAL- hecho has...,0.0,V,1.000120e+09,2026-10-23 00:00:00.0000000,None,NaN
2,1,382655104.0,103,592282.0,2026-10-23 00:00:00.0000000,Recibo creado AUTOMATICAMENTE en la Activación...,0.0,V,1.000120e+09,2026-10-23 00:00:00.0000000,26V06,NaN
3,1,382200847.0,201,592282.0,2026-10-24 00:00:00.0000000,Pago automatico por Boton de Pagos. Referencia...,76000.0,V,1.000120e+09,2026-10-24 00:00:00.0000000,26V06,2850252.0
4,1,380220011.0,201,588169.0,2026-10-05 00:00:00.0000000,Pago automatico por Boton de Pagos. Referencia...,76000.0,V,1.000120e+09,2026-10-05 00:00:00.0000000,2027A,2819459.0
...,...,...,...,...,...,...,...,...,...,...,...,...
17853,1,380355372.0,200,588058.0,2026-10-06 00:00:00.0000000,RECIBO AUTOMATICO CREADO EN EL REGISTRO DEL RE...,700000.0,V,9.846859e+07,2026-10-06 00:00:00.0000000,26V06,2822962.0
17854,1,382664772.0,103,592141.0,2026-10-23 00:00:00.0000000,Recibo creado AUTOMATICAMENTE en la Activación...,0.0,V,9.865452e+07,2026-10-23 00:00:00.0000000,26ES6,NaN
17855,1,382660345.0,201,592141.0,2026-10-26 00:00:00.0000000,Pago automatico por Boton de Pagos. Referencia...,154360.0,V,9.865452e+07,2026-10-26 00:00:00.0000000,26ES6,2852270.0
17856,1,383136300.0,103,592141.0,2026-10-29 00:00:00.0000000,Cruce AUTOMATICO DE SALDOS -ESPECIALIZACION- h...,0.0,V,9.865452e+07,2026-10-26 00:00:00.0000000,None,NaN


In [8]:
Recibos_Caja['CLIENTE'] = Recibos_Caja['CLIENTE'].fillna(0).astype(int).astype(str)
#aca filtro el dataframe solo con los pagos que corresponden a los mismo periodos que en etiquetas
Recibos_Caja['Llave_per_caja']=Recibos_Caja['CLIENTE'].astype(str)+Recibos_Caja['PERIODO'].astype(str)
Recibos_Caja = Recibos_Caja[Recibos_Caja['Llave_per_caja'].isin(Etiquetas_mes_acutual['Llave_etiq_per'])]
Recibos_Caja = Recibos_Caja.drop('Llave_per_caja', axis=1)
#---------------------------------------------------------------------------
#ACA AÑADO LAS ETIQUETAS EL VALOR PAGADO POR EL ESTUDIANTE
#---------------------------------------------------------------------------
Recibos_Caja_t=Recibos_Caja[['CLIENTE','VALOR']]
df_resumen_recibos = Recibos_Caja_t.groupby('CLIENTE', as_index=False)['VALOR'].sum()

Aqui no hay duplicados en el df de recibos caja por ende no habrá multiples coincidencias

In [9]:
#AQUÍ FORMO LA TABLA DE LO QUE TEORICAMENTE DEBIA PAGAR EL ESTUDIANTE 
#------------------------------------------------------------------------------------------------------------
Cedulas_filtroz = Etiquetas_mes_acutual['Doc_Estudiante'].unique()
cedulas_para_zoho = ", ".join([f"'{cedulaz}'" for cedulaz in Cedulas_filtroz])
#------------------------------------------------------------------------------------------------------------
host2_1 = '172.16.1.33'
port2_1 = '1433'
database2_1 = 'CUN_REPOSITORIO'
try:
    url2_1 = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host2_1, port=port2_1, db=database2_1)
    engine2_1 = create_engine(url2_1) 
    print("Conexion a la base de datos realizada")
    ventas_zoho = pd.read_sql(f'''
SELECT DISTINCT
    doc_alum as Documento_Estudiante_zoho,
	PERIODO,
	FEC_PAGO_LIQ,
    VALOR_ORDEN_IC as VALOR_ORDEN,
    val_becdtos,
    tipo,
    ADICIONALES
FROM zoho.base_personas
WHERE
   ESTADO_PAGO = 'PAGO'
   AND EST_MATRICULADO = '1-Activo'
   AND DOCUMENTO IN ('FAMA','FECO')
   and PERIODO not like '%I%'
   AND doc_alum IN ({cedulas_para_zoho})
                               ''', engine2_1)
    ventas_zoho2 = ventas_zoho.copy()
except Exception as e:
    print("Error:", e)
finally:
    if 'engine2_1' in locals():
        engine2_1.dispose()
        print("Conexión cerrada")
ventas_zoho

Conexion a la base de datos realizada
Conexión cerrada


,Documento_Estudiante_zoho,PERIODO,FEC_PAGO_LIQ,VALOR_ORDEN,val_becdtos,tipo,ADICIONALES
0,1000003468,26V06,05/09/2026,3672400,-3196400,Pregrado,24000
1,1000004751,2027A,23/10/2026,2971747,-2495747,Pregrado,24000
2,1000017479,2027A,05/09/2026,2971800,-2495800,Pregrado,24000
3,1000019263,26V06,05/09/2026,3672400,-3196400,Pregrado,24000
4,1000020216,26V06,23/10/2026,3854800,-3378800,Pregrado,24000
...,...,...,...,...,...,...,...
4587,96601727,26ES6,31/10/2026,6598000,-5278400,Postgrado,24000
4588,976287,26V06,05/09/2026,4037200,-3561200,Pregrado,24000
4589,98400070,26V06,05/09/2026,4037200,-3561200,Pregrado,24000
4590,98468588,26V06,06/10/2026,3672400,-3196400,Pregrado,24000


In [10]:
ventas_zoho['valor_ingles'] = '0'
mask = ventas_zoho['tipo'] == 'Pregrado'
ventas_zoho.loc[mask, 'valor_ingles'] = '200000'

In [11]:
ventas_zoho = ventas_zoho.drop(columns=['tipo'])

In [12]:
ventas_zoho

,Documento_Estudiante_zoho,PERIODO,FEC_PAGO_LIQ,VALOR_ORDEN,val_becdtos,ADICIONALES,valor_ingles
0,1000003468,26V06,05/09/2026,3672400,-3196400,24000,200000
1,1000004751,2027A,23/10/2026,2971747,-2495747,24000,200000
2,1000017479,2027A,05/09/2026,2971800,-2495800,24000,200000
3,1000019263,26V06,05/09/2026,3672400,-3196400,24000,200000
4,1000020216,26V06,23/10/2026,3854800,-3378800,24000,200000
...,...,...,...,...,...,...,...
4587,96601727,26ES6,31/10/2026,6598000,-5278400,24000,0
4588,976287,26V06,05/09/2026,4037200,-3561200,24000,200000
4589,98400070,26V06,05/09/2026,4037200,-3561200,24000,200000
4590,98468588,26V06,06/10/2026,3672400,-3196400,24000,200000


In [13]:
#aca filtro el dataframe solo con las matriculas que corresponden a los mismo periodos que en etiquetas
ventas_zoho['llave_per_zoho']=ventas_zoho['Documento_Estudiante_zoho'].astype(str)+ventas_zoho['PERIODO'].astype(str)
ventas_zoho = ventas_zoho[ventas_zoho['llave_per_zoho'].isin(Etiquetas_mes_acutual['Llave_etiq_per'])]
ventas_zoho = ventas_zoho.drop('llave_per_zoho', axis=1)
#--------------------------------------------------------------------
ventas_zoho['VALOR_ORDEN']=ventas_zoho['VALOR_ORDEN'].fillna(0).astype(int)
ventas_zoho['val_becdtos']=ventas_zoho['val_becdtos'].fillna(0).astype(int)
ventas_zoho['valor_ingles']=ventas_zoho['valor_ingles'].fillna(0).astype(int)
ventas_zoho['ADICIONALES']=ventas_zoho['ADICIONALES'].fillna(0).astype(int)
ventas_zoho['VALOR_A_PAGAR']=ventas_zoho['VALOR_ORDEN']+ventas_zoho['val_becdtos']+ventas_zoho['valor_ingles']+ventas_zoho['ADICIONALES']
ventas_zoho_t=ventas_zoho[['Documento_Estudiante_zoho','VALOR_ORDEN','val_becdtos','valor_ingles','ADICIONALES','VALOR_A_PAGAR']]
df_resumen_zoho = ventas_zoho_t.drop_duplicates(subset='Documento_Estudiante_zoho', keep='first')

Aca ya la tabla de ventas no tiene matriculas duplicados por ende no traera multiples coincidencias

In [14]:
#ACA AÑADO A Etiquetas_mes_actual_t TODO LO DE ZOHO y lo de Recibos Caja
#----------------------------------------------------------------
Etiquetas_mes_actual_t=Etiquetas_mes_acutual[['Doc_Estudiante','PERIODO','Documento_del_Asesor','Fecha_de_pago','Periodo_Pago','TIPO','EstadoFinal']]
df_resultado1 = pd.merge(Etiquetas_mes_actual_t, df_resumen_zoho, left_on='Doc_Estudiante', right_on='Documento_Estudiante_zoho', how='left')
df_resultado2 = pd.merge(df_resultado1, df_resumen_recibos, left_on='Doc_Estudiante', right_on='CLIENTE', how='left')
df_resultado2.drop(columns=['Documento_Estudiante_zoho','CLIENTE'], inplace=True)
#df_resultado2.to_excel('Etiquetas_ok_MesActual_netoYrecibos.xlsx') #para saber cuanto pago cada estudiante
#------------------DIVIDO ENTRE PREG, POST Y EDU----------------------------------------
df_resultado2_preg=df_resultado2[df_resultado2['TIPO']=='Pregrado'] # ACA SOLO TENGO PREGRADOS
df_resultado2_post=df_resultado2[df_resultado2['TIPO']=='Postgrado'] # ACA SOLO TENGO POSTGRADOS
df_resultado2_Ec=df_resultado2[df_resultado2['TIPO']=='Educacion continuada'] # ACA SOLO TENGO EDUC_CONTINUADA


ACA SON CALCULOS EXCLUSIVOS PARA EL DATAFRAME DE SOLO PREGRADOS

In [15]:
#-----------CALCULOS EXCLUSIVOS PREGRADO --df_resultado2_preg---------------------------------------
##df_resultado2_preg['Pago_Total'] = np.where(df_resultado2_preg['VALOR'] == df_resultado2_preg['VALOR_A_PAGAR'], 'P_Total', 'P_parcial')
df_resultado2_preg['Pago_Total'] = np.where(
    abs(df_resultado2_preg['VALOR'] - df_resultado2_preg['VALOR_A_PAGAR']) <= 1000,
    'P_Total',
    'P_parcial'
)
df_resultado2_preg['Modalidad'] = df_resultado2_preg['PERIODO'].apply(lambda x: 'VIRTUAL' if 'V' in x else 'PRESENCIAL')
##df_resultado2_preg.to_excel('CuantoPago_Etiq_SoloPreg.xlsx')
almacen_etiquetas_preg_aprov=df_resultado2_preg.copy()
almacen_etiquetas_preg_aprov['Periodo_Pago_1']=PERIODO_ALMACEN
#--------------------------------------------------------------------------------------------------
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'PREGR_APROB_CONTAC_MOD2'
schema_name_2_12 = 'COMISIONES_CONTACT'
almacen_etiquetas_preg_aprov.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_2028\1081943013.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_resultado2_preg['Pago_Total'] = np.where(
C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_2028\1081943013.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_resultado2_preg['Modalidad'] = df_resultado2_preg['PERIODO'].apply(lambda x: 'VIRTUAL' if 'V' in x else 'PRESENCIAL')


In [16]:
df_resultado2_preg['Llave_venta']=df_resultado2_preg['Documento_del_Asesor'].astype(str)+df_resultado2_preg['Pago_Total'].astype(str)+df_resultado2_preg['Modalidad'].astype(str)

#ACA TOMO EL DATAFRAME DE PREGRADOS Y HAGO LOS CONTEOS POR MODALIDAD Y PAGO TOTAL Y LO METO LOS CONTEOS EN UNO NUEVO
#--------------------------------------------------------------------------------------------------------------------
conteo_documentos = df_resultado2_preg['Documento_del_Asesor'].value_counts().reset_index()
conteo_documentos.columns = ['Documento_del_Asesor', 'Frecuencia']
def contar_condiciones(df, documento, pago_total, modalidad):
    return len(df[(df['Documento_del_Asesor'] == documento) & (df['Pago_Total'] == pago_total) & (df['Modalidad'] == modalidad)])
conteo_documentos['P_parcial-PRESENCIAL'] = conteo_documentos['Documento_del_Asesor'].apply(lambda x: contar_condiciones(df_resultado2_preg, x, 'P_parcial', 'PRESENCIAL'))
conteo_documentos['P_Total-PRESENCIAL'] = conteo_documentos['Documento_del_Asesor'].apply(lambda x: contar_condiciones(df_resultado2_preg, x, 'P_Total', 'PRESENCIAL'))
conteo_documentos['P_parcial-VIRTUAL'] = conteo_documentos['Documento_del_Asesor'].apply(lambda x: contar_condiciones(df_resultado2_preg, x, 'P_parcial', 'VIRTUAL'))
conteo_documentos['P_Total-VIRTUAL'] = conteo_documentos['Documento_del_Asesor'].apply(lambda x: contar_condiciones(df_resultado2_preg, x, 'P_Total', 'VIRTUAL'))
#ACA HAGO LOS CALCULOS PARA LLENAR LAS COLUMNAS CON LAS MULTIPLICACIONES
#---------------------------------------------------------------------------------------
conteo_documentos['Vent_Parc_PRES'] = 0
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 30) & (conteo_documentos['Frecuencia'] < 40), 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 100000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 40) & (conteo_documentos['Frecuencia'] < 50), 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 110000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 50) & (conteo_documentos['Frecuencia'] < 60), 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 120000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 60) & (conteo_documentos['Frecuencia'] < 70), 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 130000
conteo_documentos.loc[conteo_documentos['Frecuencia'] >= 70, 'Vent_Parc_PRES'] = conteo_documentos['P_parcial-PRESENCIAL'] * 140000
conteo_documentos['Vent_Total_PRES'] = 0
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 30) & (conteo_documentos['Frecuencia'] < 40), 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 150000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 40) & (conteo_documentos['Frecuencia'] < 50), 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 165000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 50) & (conteo_documentos['Frecuencia'] < 60), 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 180000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 60) & (conteo_documentos['Frecuencia'] < 70), 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 195000
conteo_documentos.loc[conteo_documentos['Frecuencia'] >= 70, 'Vent_Total_PRES'] = conteo_documentos['P_Total-PRESENCIAL'] * 210000
conteo_documentos['Vent_Parc_VIRT'] = 0
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 30) & (conteo_documentos['Frecuencia'] < 40), 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 45000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 40) & (conteo_documentos['Frecuencia'] < 50), 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 55000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 50) & (conteo_documentos['Frecuencia'] < 60), 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 65000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 60) & (conteo_documentos['Frecuencia'] < 70), 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 75000
conteo_documentos.loc[conteo_documentos['Frecuencia'] >= 70, 'Vent_Parc_VIRT'] = conteo_documentos['P_parcial-VIRTUAL'] * 85000
conteo_documentos['Vent_Total_VIRT'] = 0
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 30) & (conteo_documentos['Frecuencia'] < 40), 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 80000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 40) & (conteo_documentos['Frecuencia'] < 50), 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 100000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 50) & (conteo_documentos['Frecuencia'] < 60), 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 120000
conteo_documentos.loc[(conteo_documentos['Frecuencia'] >= 60) & (conteo_documentos['Frecuencia'] < 70), 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 140000
conteo_documentos.loc[conteo_documentos['Frecuencia'] >= 70, 'Vent_Total_VIRT'] = conteo_documentos['P_Total-VIRTUAL'] * 150000
conteo_documentos['Total_Com_Preg']= conteo_documentos['Vent_Parc_PRES'] + conteo_documentos['Vent_Total_PRES'] + conteo_documentos['Vent_Parc_VIRT'] + conteo_documentos['Vent_Total_VIRT']
#conteo_documentos.to_excel('Tabla_conteos_preg_total.xlsx')

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_2028\4103060226.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_resultado2_preg['Llave_venta']=df_resultado2_preg['Documento_del_Asesor'].astype(str)+df_resultado2_preg['Pago_Total'].astype(str)+df_resultado2_preg['Modalidad'].astype(str)


In [17]:
#-----------CALCULOS EXCLUSIVOS POST--df_resultado2_post---------------------------------------
##df_resultado2_post['Pago_Total'] = np.where(df_resultado2_post['VALOR'] == df_resultado2_post['VALOR_A_PAGAR'], 'P_Total', 'P_parcial')
df_resultado2_post['Pago_Total'] = np.where(
    abs(df_resultado2_post['VALOR'] - df_resultado2_post['VALOR_A_PAGAR']) <= 1000,
    'P_Total',
    'P_parcial'
)
##df_resultado2_preg.to_excel('CuantoPago_Etiq_SoloPreg.xlsx')
almacen_etiquetas_post_aprov=df_resultado2_post.copy()
almacen_etiquetas_post_aprov['Periodo_Pago_1']=PERIODO_ALMACEN

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_2028\3384217901.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_resultado2_post['Pago_Total'] = np.where(


In [18]:
#SUBO LA DATA DE POSGRADOS APROBADOS

host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'POSG_APROB_CONTAC_MOD2'
schema_name_2_12 = 'COMISIONES_CONTACT'
almacen_etiquetas_post_aprov.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()

In [19]:


#ACA TOMO EL DATAFRAME DE POSGRADOS Y HAGO LOS CONTEOS POR PAGO TOTAL Y LO METO LOS CONTEOS EN UNO NUEVO
#--------------------------------------------------------------------------------------------------------------------
#-----------------------------------------------------
conteo_documentos_post = df_resultado2_post['Documento_del_Asesor'].value_counts().reset_index()
conteo_documentos_post.columns = ['Documento_del_Asesor_post', 'Ventas_post']
def contar_condiciones_pos(df, documento, pago_total):
    return len(df[(df['Documento_del_Asesor'] == documento) & (df['Pago_Total'] == pago_total)])
conteo_documentos_post['P_parcial'] = conteo_documentos_post['Documento_del_Asesor_post'].apply(lambda x: contar_condiciones_pos(df_resultado2_post, x, 'P_parcial'))
conteo_documentos_post['P_Total'] = conteo_documentos_post['Documento_del_Asesor_post'].apply(lambda x: contar_condiciones_pos(df_resultado2_post, x, 'P_Total'))

#ACA HAGO LOS CALCULOS PARA LLENAR LAS COLUMNAS CON LAS MULTIPLICACIONES
#---------------------------------------------------------------------------------------
conteo_documentos_post['Vent_Parc'] = 0
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 30) & (conteo_documentos_post['Ventas_post'] < 40), 'Vent_Parc'] = conteo_documentos_post['P_parcial'] * 50400
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 40) & (conteo_documentos_post['Ventas_post'] < 50), 'Vent_Parc'] = conteo_documentos_post['P_parcial'] * 64800
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 50) & (conteo_documentos_post['Ventas_post'] < 60), 'Vent_Parc'] = conteo_documentos_post['P_parcial'] * 79200
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 60) & (conteo_documentos_post['Ventas_post'] < 70), 'Vent_Parc'] = conteo_documentos_post['P_parcial'] * 93600
conteo_documentos_post.loc[conteo_documentos_post['Ventas_post'] >= 70, 'Vent_Parc'] = conteo_documentos_post['P_parcial'] * 108000
conteo_documentos_post['Vent_Total'] = 0
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 30) & (conteo_documentos_post['Ventas_post'] < 40), 'Vent_Total'] = conteo_documentos_post['P_Total'] * 112000
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 40) & (conteo_documentos_post['Ventas_post'] < 50), 'Vent_Total'] = conteo_documentos_post['P_Total'] * 144000
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 50) & (conteo_documentos_post['Ventas_post'] < 60), 'Vent_Total'] = conteo_documentos_post['P_Total'] * 176000
conteo_documentos_post.loc[(conteo_documentos_post['Ventas_post'] >= 60) & (conteo_documentos_post['Ventas_post'] < 70), 'Vent_Total'] = conteo_documentos_post['P_Total'] * 208000
conteo_documentos_post.loc[conteo_documentos_post['Ventas_post'] >= 70, 'Vent_Total'] = conteo_documentos_post['P_Total'] * 240000
conteo_documentos_post['Total_Com_Pos']= conteo_documentos_post['Vent_Parc'] + conteo_documentos_post['Vent_Total'] 

In [20]:
#CALCULO DE POSTGRADOS ANTIGUO (VIEJO MARZO)
#-----------------------------------------------------
#conteo_documentos_post = df_resultado2_post['Documento_del_Asesor'].value_counts().reset_index()
#conteo_documentos_post.columns = ['Documento_del_Asesor_post', 'Ventas_post']
#conteo_documentos_post.to_excel('VentasPost.xlsx')
#def calcular_valor_ventas(cantidad):
#    if cantidad == 0:
#        return 0
#    elif 1 <= cantidad < 6:
#        return cantidad * 20000
#    elif 6 <= cantidad < 11:
#        return cantidad * 40000
#    elif 11 <= cantidad < 21:
#        return cantidad * 60000
#    elif cantidad >= 21:
#        return cantidad * 90000
#conteo_documentos_post['Valor_Vent_post'] = conteo_documentos_post['Ventas_post'].apply(calcular_valor_ventas)
#conteo_documentos_post.to_excel('VentasPostN.xlsx')
#-----------------------------------------------------
#CALCULO EDUCACION CONTINUADA
#-----------------------------------------------------
conteo_documentos_Ec = df_resultado2_Ec['Documento_del_Asesor'].value_counts().reset_index()
conteo_documentos_Ec.columns = ['Documento_del_Asesor_Ec', 'Ventas_Ec']
#conteo_documentos_Ec.to_excel('Ventas_Ec.xlsx')
def calcular_valor_ventas_Ec(cantidad):
    if cantidad == 0:
        return 0
    elif cantidad >= 1:
        return cantidad * 10000
conteo_documentos_Ec['Valor_Vent_Ec'] = conteo_documentos_Ec['Ventas_Ec'].apply(calcular_valor_ventas_Ec)
#conteo_documentos_Ec.to_excel('Ventas_EcN.xlsx')

In [21]:
#ACA AÑADO AL DATAFRAME DE PREGRADOS POSTGRADOS
#------------------------------------------------------------------
resultado_m = pd.merge(conteo_documentos, conteo_documentos_post, left_on='Documento_del_Asesor', right_on='Documento_del_Asesor_post', how='outer')
resultado_m['Documento_Combinado'] = resultado_m['Documento_del_Asesor'].combine_first(resultado_m['Documento_del_Asesor_post'])
resultado_m.drop(columns=['Documento_del_Asesor', 'Documento_del_Asesor_post'], inplace=True)
resultado_m=resultado_m.fillna(0)
columnas = resultado_m.columns.tolist()
columnas = ['Documento_Combinado'] + [col for col in columnas if col != 'Documento_Combinado']
resultado_m = resultado_m[columnas]
resultado_m.rename(columns={'Documento_Combinado': 'Documento_del_Asesor'}, inplace=True)
#resultado_m.to_excel('Merge_totalC_C.xlsx')
#------------------------------------------------------------------
#ACA AÑADO AL DATAFRAME DE UNIDO EDUCACION CONTINUADA
#------------------------------------------------------------------
resultado_f = pd.merge(resultado_m, conteo_documentos_Ec, left_on='Documento_del_Asesor', right_on='Documento_del_Asesor_Ec', how='outer')
resultado_f['Documento_Combinado'] = resultado_f['Documento_del_Asesor'].combine_first(resultado_f['Documento_del_Asesor_Ec'])
resultado_f.drop(columns=['Documento_del_Asesor', 'Documento_del_Asesor_Ec'], inplace=True)
resultado_f=resultado_f.fillna(0)
columnas1 = resultado_f.columns.tolist()
columnas1 = ['Documento_Combinado'] + [col1 for col1 in columnas1 if col1 != 'Documento_Combinado']
resultado_f = resultado_f[columnas1]
#------------------------------------------------------------------
#ENCUENTRO EL TOTAL TOTAL
#------------------------------------------------------------------
resultado_f['TOTAL_COMISION_GENERAL']=resultado_f['Total_Com_Preg']+resultado_f['Total_Com_Pos']+resultado_f['Valor_Vent_Ec']
##resultado_f.to_excel('Merge_totalC_f.xlsx') #Ya tengo aqui mi tabla de ventas totales


In [22]:
resultado_f.rename(columns={'Total_Com_Pos':'Valor_Vent_post','P_parcial':'P_parcial_posg','P_Total':'P_Total_posg',
                            'Vent_Parc':'Vent_Parc_posg', 'Vent_Total' : 'Vent_Total_posg'}, inplace=True)

In [23]:
#AQUI INYECTO LA BASE DESGLOSADA COMPLETA POR QUE CUANDO EJECUTO EL PYTHON DE RELIQUIDACIONES SE NECESITA
resultado_almacen=resultado_f.copy()
resultado_almacen['Periodo_Pago']=PERIODO_ALMACEN
#-------------------------------------------------------------------
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'FIJA_LIQ_ACTUAL_DESGLOS'
schema_name_2_12 = 'COMISIONES_CONTACT'
resultado_almacen.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()

In [24]:
#AQUI YA FORMO LA TABLA QUE NECESITA EL SP
#------------------------------------------------
Tabla_Conteos_SP=resultado_f.copy()
Tabla_Conteos_SP['Periodo_Pago']=PERIODO_NORMAL_float
Tabla_Conteos_SP=Tabla_Conteos_SP[['Documento_Combinado','Periodo_Pago','Frecuencia','Ventas_post','Ventas_Ec','Total_Com_Preg','Valor_Vent_post','Valor_Vent_Ec','TOTAL_COMISION_GENERAL']] 
Tabla_Conteos_SP.rename(columns={'Documento_Combinado': 'Documento_del_Asesor',
                            'Frecuencia': 'Pregrados',
                            'Ventas_post': 'Postgrado',
                            'Ventas_Ec': 'Educacion_continuada',
                            'Total_Com_Preg': 'Valor_Pregrado',
                            'Valor_Vent_post': 'Valor_Postgrado',
                            'Valor_Vent_Ec': 'Valor_Educacion_continuada',
                            'TOTAL_COMISION_GENERAL': 'Total_Comision',
                            }, inplace=True)
#Tabla_Conteos_SP.to_excel('Tabla conteos pal sp.xlsx')#Esta es la tabla que se inyecta a SQL

In [25]:
Tabla_Conteos_SP.dtypes
#AQUI SIMPLEMENTE VALIDE QUE TIPOS DE DATOSE IBA A METER A LA TABLA
# df_dtypesTabla_Conteos_SP = pd.DataFrame(Tabla_Conteos_SP.dtypes).reset_index()
# df_dtypesTabla_Conteos_SP.columns = ['Columna', 'Tipo de Dato']
# df_dtypesTabla_Conteos_SP.to_excel('TIPOS CONTEOS.xlsx')

Documento_del_Asesor           object
Periodo_Pago                   object
Pregrados                     float64
Postgrado                     float64
Educacion_continuada          float64
Valor_Pregrado                float64
Valor_Postgrado               float64
Valor_Educacion_continuada    float64
Total_Comision                float64
dtype: object

In [26]:
host_2_12 = '172.16.1.33'
port_2_12 = '1433'
database_2_12 = 'CUN_REPOSITORIO'
url_2_12 = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host_2_12, port=port_2_12, db=database_2_12)
engine_2_12 = create_engine(url_2_12, fast_executemany=True)

table_name_2_12 = 'TEMP_CONTEO_VENTAS'
schema_name_2_12 = 'COMISIONES_CONTACT'
# Send the data to sql server
Tabla_Conteos_SP.to_sql(
    name=table_name_2_12,
    schema=schema_name_2_12,
    con=engine_2_12,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine_2_12.dispose()

In [27]:
#CONSULTA PARA CREAR LA TABLA TEMPORAL EN SQL 
with engine.connect() as connection:
    try:

        sql_query = text(f'''
        IF object_id('CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_AFECTACIONES') IS NOT NULL
        DROP TABLE CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_AFECTACIONES

        Select *, Calidad_Resultado+Formacion_Resultado+Ventas_Resultado+Adherencia_Resultado+Conversion_Resultado+Productividad_Resultado AS Por_Afectacion
        INTO CUN_REPOSITORIO.COMISIONES_CONTACT.TEMP_AFECTACIONES 
        from(
            Select Documento_del_Asesor,Correo_Asesor,Periodo,
            case when CALIDAD <> 'Cumple' then 0.07 else 0 end as Calidad_Resultado,
            case when [FORMACION ]  <> 'Cumple' then 0.07 else 0 end as Formacion_Resultado,
            case when VENTAS  <> 'Cumple' then 0.07 else 0 end as Ventas_Resultado,
            case when ADHERENCIA  <> 'Cumple' then 0.07 else 0 end as Adherencia_Resultado,
            case when CONVERSION  <> 'Cumple' then 0.07 else 0 end as Conversion_Resultado,
            case when PRODUCTIVIDAD  <> 'Cumple' then 0.07 else 0 end as Productividad_Resultado
        from CUN_REPOSITORIO.Comisiones_Contact.Afectaciones_Contact)t1
        ''')
        connection.execute(sql_query)
        connection.commit()  # Asegurar que los cambios se guarden en la base de datos
        print("Consulta realizada con exito")
    except Exception as e:
        print(f"Error al eliminar registros: {e}")
    finally:
        connection.close()

Consulta realizada con exito
